In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [2]:
# 1. DATASET LOADING & CLEANING
# ==========================================
print("Loading Telco Churn Dataset...")
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

Loading Telco Churn Dataset...


In [3]:
# Drop unique identifier column as it holds no predictive power
df.drop(columns=['customerID'], inplace=True, errors='ignore')

In [4]:
# Handle empty string discrepancies in TotalCharges column safely
df['TotalCharges'] = df['TotalCharges'].replace(" ", np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

/tmp/ipykernel_1735/2030144771.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)


In [5]:
# Encode Target variable 'Churn' to binary integers (Yes -> 1, No -> 0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

X = df.drop(columns=['Churn'])
y = df['Churn']

In [6]:
# Identify numeric vs categorical input tracks automatically
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

In [7]:
# Split into structured Train and Test pools
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [8]:
# 2. MACHINE LEARNING PIPELINE BUILDING
# ==========================================
print("Constructing data preprocessing transformers...")

# Define processing strategies for numerical data
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

Constructing data preprocessing transformers...


In [9]:
# Define processing strategies for textual categories
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [10]:
# Merge both transformations into a single data preprocessor grid
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [11]:
# Combine preprocessor with a baseline classifier model into one fluid Pipeline structure
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [12]:
# 3. HYPERPARAMETER TUNING (GRID SEARCH)
# ==========================================
print("Launching GridSearchCV hyperparameter optimization...")

Launching GridSearchCV hyperparameter optimization...


In [13]:
# Setup search space targeting Random Forest parameters
param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [5, 10, None],
}

grid_search = GridSearchCV(full_pipeline, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Optimal Parameters Found: {grid_search.best_params_}")

Optimal Parameters Found: {'classifier__max_depth': 10, 'classifier__n_estimators': 100}


In [14]:
# 4. EVALUATION & MODEL EXPORT
# ==========================================
# Predict using the optimized best configuration pipeline wrapper directly
best_pipeline = grid_search.best_estimator_
y_pred = best_pipeline.predict(X_test)

In [15]:
print("\n--- Pipeline Evaluation Metrics Summary ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print("\nDetailed Classification Matrix:")
print(classification_report(y_test, y_pred))


--- Pipeline Evaluation Metrics Summary ---
Accuracy Score: 0.8027

Detailed Classification Matrix:
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.66      0.53      0.59       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.79      0.80      0.80      1409



In [16]:
# Serialized deployment export block
model_filename = "telco_churn_pipeline.joblib"
joblib.dump(best_pipeline, model_filename)
print(f"Success! Complete production pipeline exported to: '{model_filename}'")

Success! Complete production pipeline exported to: 'telco_churn_pipeline.joblib'
